# 자유 실습: 이제 여러분의 데이터입니다

체험 1 > 체험 2 > 체험 3 > **자유 실습**

---

지금까지 화학 공정 데이터로 연습했습니다.
여러분 업종에 맞는 데이터로 바꿔보세요.

> **먼저 아래 [준비] 셀을 실행하세요.**
> 그 다음 드롭다운에서 업종을 선택하고 **데이터 변경** 버튼을 누르세요.

| 업종 | X (공정조건) | Y (품질) |
|:----:|-------------|----------|
| 화학 공정 | 온도, 압력, 유량, 교반, 시간 | 순도, 입자크기, 점도 |
| 사출 성형 | 사출압, 보압, 금형온도, 냉각시간, 수지온도 | 치수정밀도, 수축률, 외관등급 |
| CNC 정밀가공 | 절삭속도, 이송률, 절삭깊이, 공구마모, 냉각유량 | 표면조도, 치수편차, 가공시간 |
| 식품/바이오 | 배양온도, pH, 배양시간, 교반속도, 원료농도 | 생균수, 안정성, 수율 |
| 범용 제조 | 설비온도, 가동속도, 원료투입, 압력, 작업시간 | 불량률, 수율, 에너지효율 |

---

## Colab AI에게 시켜볼 것들

데이터 질의:

```
순도가 97% 이상인 로트만 필터링해서 보여줘
```

시각화:

```
온도별 평균 순도를 막대 그래프로 그려줘
```

```
각 품질 항목의 boxplot을 그려줘
```

```
온도를 50에서 200까지 바꿀 때 순도가 어떻게 변하는지 그래프로 보여줘
```

리포트:

```
각 품질 항목의 평균, 표준편차, 규격 이탈 비율을 표로 정리해줘
```

---

## 회사에 돌아가면

이 노트북에 여러분 MES 데이터(CSV)를 넣으면 오늘 한 것을 그대로 반복할 수 있습니다.

```
1. CSV 파일을 Colab에 업로드
2. "이 CSV를 df로 읽어줘"
3. "df에서 X열과 Y열을 알려줘"
4. "Random Forest로 모델을 만들어줘"
5. "온도 150일 때 품질 예측해줘"
```

**오늘 했던 것, 그대로입니다.**


In [ ]:
#@title **[준비] 실행 버튼(>)만 눌러주세요** (코드를 읽을 필요 없습니다)
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

_ws = {'description_width':'120px'}
_wl = widgets.Layout(width='500px')

def _gen(seed, x_cfg, y_funcs, n=1000):
    rng = np.random.default_rng(seed)
    xs = {nm: rng.uniform(lo, hi, n) for nm, (lo, hi) in x_cfg.items()}
    ys = {nm: fn(xs, rng) for nm, fn in y_funcs.items()}
    return pd.DataFrame({**xs, **ys})

DATASETS = {}

DATASETS['화학 공정 (기본)'] = dict(
    x_cfg={'온도_C': (50,200), '압력_kPa': (100,500), '유량_Lmin': (1,10), '교반_RPM': (100,800), '시간_min': (10,120)},
    y_funcs={
        '순도_pct': lambda x, r: np.clip(80+0.05*x['온도_C']+0.01*x['압력_kPa']-0.3*x['유량_Lmin']+0.005*x['교반_RPM']+0.02*x['시간_min']-0.0001*x['온도_C']*x['압력_kPa']+0.0002*x['온도_C']*x['교반_RPM']+r.normal(0,1.5,1000), 85, 99.9),
        '입자크기_um': lambda x, r: np.clip(25-0.05*x['온도_C']+0.01*x['압력_kPa']+0.8*x['유량_Lmin']-0.01*x['교반_RPM']-0.03*x['시간_min']+0.0005*x['유량_Lmin']*x['교반_RPM']+r.normal(0,1.0,1000), 1, 50),
        '점도_cP': lambda x, r: np.clip(50+0.3*x['온도_C']+0.1*x['압력_kPa']-2.0*x['유량_Lmin']+0.05*x['교반_RPM']+0.5*x['시간_min']-0.001*x['온도_C']*x['유량_Lmin']+r.normal(0,5,1000), 30, 400),
    },
    specs={'순도_pct': (95,99.9,'순도(%)'), '입자크기_um': (5,20,'입자크기(um)'), '점도_cP': (80,250,'점도(cP)')},
    seed=42,
)

DATASETS['사출 성형'] = dict(
    x_cfg={'사출압_bar': (50,200), '보압_bar': (30,120), '금형온도_C': (30,100), '냉각시간_sec': (5,60), '수지온도_C': (180,280)},
    y_funcs={
        '치수정밀도_mm': lambda x, r: np.clip(0.5-0.002*x['사출압_bar']+0.001*x['보압_bar']-0.003*x['금형온도_C']+0.005*x['냉각시간_sec']+0.001*x['수지온도_C']-0.00002*x['사출압_bar']*x['금형온도_C']+r.normal(0,0.02,1000), 0.01, 1.0),
        '수축률_pct': lambda x, r: np.clip(3.0-0.005*x['사출압_bar']-0.01*x['보압_bar']+0.02*x['금형온도_C']-0.01*x['냉각시간_sec']+0.005*x['수지온도_C']+r.normal(0,0.15,1000), 0.1, 5.0),
        '외관등급': lambda x, r: np.clip(8.0+0.005*x['사출압_bar']+0.01*x['보압_bar']-0.02*x['금형온도_C']+0.01*x['냉각시간_sec']-0.005*x['수지온도_C']+0.0001*x['사출압_bar']*x['보압_bar']+r.normal(0,0.3,1000), 1, 10),
    },
    specs={'치수정밀도_mm': (0.05,0.3,'치수정밀도(mm)'), '수축률_pct': (0.5,2.0,'수축률(%)'), '외관등급': (7,10,'외관등급(1-10)')},
    seed=43,
)

DATASETS['CNC 정밀가공'] = dict(
    x_cfg={'절삭속도_mpm': (50,300), '이송률_mmrev': (0.05,0.5), '절삭깊이_mm': (0.1,5.0), '공구마모_pct': (0,80), '냉각유량_Lmin': (1,10)},
    y_funcs={
        '표면조도_Ra': lambda x, r: np.clip(2.0-0.003*x['절삭속도_mpm']+3.0*x['이송률_mmrev']+0.2*x['절삭깊이_mm']+0.01*x['공구마모_pct']-0.05*x['냉각유량_Lmin']+0.05*x['이송률_mmrev']*x['공구마모_pct']+r.normal(0,0.1,1000), 0.1, 6.0),
        '치수편차_um': lambda x, r: np.clip(10-0.01*x['절삭속도_mpm']+5.0*x['이송률_mmrev']+1.5*x['절삭깊이_mm']+0.08*x['공구마모_pct']-0.3*x['냉각유량_Lmin']+r.normal(0,1.0,1000), 0.5, 30),
        '가공시간_sec': lambda x, r: np.clip(120-0.2*x['절삭속도_mpm']-50*x['이송률_mmrev']+15*x['절삭깊이_mm']+0.1*x['공구마모_pct']+0.5*x['냉각유량_Lmin']+r.normal(0,3,1000), 10, 300),
    },
    specs={'표면조도_Ra': (0.2,1.6,'표면조도(Ra)'), '치수편차_um': (1,10,'치수편차(um)'), '가공시간_sec': (20,90,'가공시간(sec)')},
    seed=44,
)

DATASETS['식품/바이오'] = dict(
    x_cfg={'배양온도_C': (25,42), 'pH': (4.0,8.0), '배양시간_hr': (12,72), '교반속도_RPM': (50,300), '원료농도_pct': (1,10)},
    y_funcs={
        '생균수_logCFU': lambda x, r: np.clip(6.0+0.1*x['배양온도_C']-0.3*np.abs(x['pH']-6.5)+0.03*x['배양시간_hr']+0.002*x['교반속도_RPM']+0.05*x['원료농도_pct']-0.001*x['배양온도_C']*np.abs(x['pH']-6.5)+r.normal(0,0.2,1000), 5, 11),
        '안정성_month': lambda x, r: np.clip(18-0.2*x['배양온도_C']+0.5*(7-np.abs(x['pH']-6.5))-0.05*x['배양시간_hr']+0.01*x['교반속도_RPM']-0.1*x['원료농도_pct']+r.normal(0,1.0,1000), 1, 36),
        '수율_pct': lambda x, r: np.clip(40+0.5*x['배양온도_C']-2*np.abs(x['pH']-6.5)+0.3*x['배양시간_hr']+0.02*x['교반속도_RPM']+1.5*x['원료농도_pct']-0.01*x['배양시간_hr']*x['원료농도_pct']+r.normal(0,2,1000), 10, 95),
    },
    specs={'생균수_logCFU': (8,10.5,'생균수(logCFU)'), '안정성_month': (12,30,'안정성(개월)'), '수율_pct': (60,90,'수율(%)')},
    seed=45,
)

DATASETS['범용 제조'] = dict(
    x_cfg={'설비온도_C': (30,250), '가동속도_rpm': (100,1500), '원료투입_kg': (10,100), '압력_bar': (1,20), '작업시간_min': (5,120)},
    y_funcs={
        '불량률_pct': lambda x, r: np.clip(8-0.01*x['설비온도_C']-0.001*x['가동속도_rpm']+0.02*x['원료투입_kg']-0.1*x['압력_bar']+0.01*x['작업시간_min']+0.00005*x['설비온도_C']*x['가동속도_rpm']+r.normal(0,0.5,1000), 0.1, 15),
        '수율_pct': lambda x, r: np.clip(70+0.03*x['설비온도_C']+0.005*x['가동속도_rpm']-0.05*x['원료투입_kg']+0.2*x['압력_bar']-0.02*x['작업시간_min']+r.normal(0,2,1000), 50, 99),
        '에너지효율_pct': lambda x, r: np.clip(85-0.02*x['설비온도_C']-0.005*x['가동속도_rpm']+0.01*x['원료투입_kg']+0.1*x['압력_bar']-0.05*x['작업시간_min']+r.normal(0,1.5,1000), 40, 99),
    },
    specs={'불량률_pct': (0.5,3.0,'불량률(%)'), '수율_pct': (80,98,'수율(%)'), '에너지효율_pct': (70,95,'에너지효율(%)')},
    seed=46,
)

def load_dataset(name):
    global df, input_cols, output_cols, factory_model, _specs, _bounds, _fi, _fi_names, _r2
    cfg = DATASETS[name]
    df = _gen(cfg['seed'], cfg['x_cfg'], cfg['y_funcs'])
    input_cols = list(cfg['x_cfg'].keys())
    output_cols = list(cfg['y_funcs'].keys())
    _specs = cfg['specs']
    _bounds = tuple((lo, hi) for lo, hi in cfg['x_cfg'].values())
    _fi_names = {c: c.split('_')[0] for c in input_cols}
    X_tr, X_te, y_tr, y_te = train_test_split(df[input_cols], df[output_cols], test_size=0.2, random_state=42)
    factory_model = RandomForestRegressor(n_estimators=100, random_state=42)
    factory_model.fit(X_tr, y_tr)
    y_pred = factory_model.predict(X_te)
    _r2 = {col: r2_score(y_te.iloc[:,i], y_pred[:,i]) for i, col in enumerate(output_cols)}
    _fi = np.mean([t.feature_importances_ for t in factory_model.estimators_], axis=0)

load_dataset('화학 공정 (기본)')

def find_optimal(targets, n_restarts=30):
    t_arr = np.array([targets[c] for c in output_cols])
    scales = np.array([max(_specs[c][1]-_specs[c][0], 1) for c in output_cols])
    def obj(x):
        return np.sqrt(np.sum(((factory_model.predict([x])[0]-t_arr)/scales)**2))
    best = None
    for _ in range(n_restarts):
        x0 = [np.random.uniform(lo,hi) for lo,hi in _bounds]
        res = minimize(obj, x0, bounds=_bounds, method='L-BFGS-B')
        if best is None or res.fun < best.fun: best = res
    return best.x, factory_model.predict([best.x])[0]

def _forward_widget():
    sliders = []
    for col in input_cols:
        lo, hi = _bounds[input_cols.index(col)]
        nm = _fi_names[col]
        if (hi - lo) > 10:
            sl = widgets.IntSlider(value=int((lo+hi)/2),min=int(lo),max=int(hi),step=max(1,int((hi-lo)/40)),description=f'{nm}:',style=_ws,layout=_wl)
        else:
            sl = widgets.FloatSlider(value=round((lo+hi)/2,2),min=lo,max=hi,step=round((hi-lo)/40,3),description=f'{nm}:',style=_ws,layout=_wl)
        sliders.append(sl)
    out = widgets.HTML()
    def up(*_):
        vals = [s.value for s in sliders]
        p = factory_model.predict([vals])[0]
        rows = ''
        all_ok = True
        for i, col in enumerate(output_cols):
            lo,hi,nm = _specs[col]
            v = p[i]; ok = lo<=v<=hi
            if not ok: all_ok = False
            clr = '#16a34a' if ok else '#dc2626'
            lb = 'OK' if ok else 'FAIL'
            rows += f'<tr><td style="padding:4px 12px">{nm}</td><td style="padding:4px 12px"><b>{v:.2f}</b></td><td style="padding:4px 12px">{lo}~{hi}</td><td style="padding:4px 12px;color:{clr};font-weight:bold">{lb}</td></tr>'
        sc = '#16a34a' if all_ok else '#dc2626'
        msg = '전 항목 규격 이내' if all_ok else '규격 이탈 항목 있음'
        out.value = (f'<div style="margin:12px 0;padding:12px 16px;background:#f0f7ff;border-left:4px solid {sc};border-radius:6px;font-size:14px">'
            f'<b style="color:{sc}">{msg}</b>'
            '<table style="margin-top:8px;border-collapse:collapse">'
            '<tr style="border-bottom:2px solid #cbd5e1"><th style="padding:4px 12px;text-align:left">품질</th><th style="padding:4px 12px;text-align:left">예측값</th><th style="padding:4px 12px;text-align:left">규격</th><th style="padding:4px 12px;text-align:left">판정</th></tr>'
            f'{rows}</table></div>')
    for s in sliders: s.observe(up,'value')
    up()
    return widgets.VBox(sliders + [out])

def _inverse_widget():
    inputs = []
    for col in output_cols:
        lo,hi,nm = _specs[col]
        inp = widgets.FloatText(value=round((lo+hi)/2,1),description=f'{nm}:',style=_ws,layout=widgets.Layout(width='300px'))
        inputs.append(inp)
    btn = widgets.Button(description='최적 조건 탐색',button_style='primary',layout=widgets.Layout(width='200px',height='36px'))
    out = widgets.HTML(value='<p style="color:#64748b">목표 품질을 입력하고 버튼을 눌러주세요.</p>')
    def click(b):
        out.value = '<p style="color:#3B82F6"><b>탐색 중...</b></p>'
        tgt = {col: inp.value for col, inp in zip(output_cols, inputs)}
        opt_x, opt_p = find_optimal(tgt)
        cr = ''.join(f'<tr><td style="padding:3px 10px">{_fi_names[col]}</td><td style="padding:3px 10px"><b>{opt_x[i]:.2f}</b></td></tr>' for i, col in enumerate(input_cols))
        pr = ''
        for i, col in enumerate(output_cols):
            lo,hi,nm = _specs[col]
            v = opt_p[i]; d = v - inputs[i].value
            ok = lo<=v<=hi
            clr = '#16a34a' if ok else '#dc2626'
            pr += f'<tr><td style="padding:3px 10px">{nm}</td><td style="padding:3px 10px"><b>{v:.2f}</b></td><td style="padding:3px 10px">{inputs[i].value}</td><td style="padding:3px 10px;color:{clr}">{d:+.2f}</td></tr>'
        out.value = ('<div style="margin:12px 0;padding:12px 16px;background:#f0fdf4;border-left:4px solid #16a34a;border-radius:6px;font-size:14px">'
            '<b>최적 공정조건</b>'
            f'<table style="margin:8px 0;border-collapse:collapse">{cr}</table>'
            '<b>예측 품질 vs 목표</b>'
            '<table style="margin:8px 0;border-collapse:collapse"><tr style="border-bottom:1px solid #cbd5e1"><th style="padding:3px 10px;text-align:left">품질</th><th style="padding:3px 10px;text-align:left">예측</th><th style="padding:3px 10px;text-align:left">목표</th><th style="padding:3px 10px;text-align:left">차이</th></tr>'
            f'{pr}</table></div>')
    btn.on_click(click)
    return widgets.VBox(inputs + [btn, out])

def rebuild_widgets():
    global forward_ui, inverse_ui
    forward_ui = _forward_widget()
    inverse_ui = _inverse_widget()

forward_ui = _forward_widget()
inverse_ui = _inverse_widget()

_selector = widgets.Dropdown(options=list(DATASETS.keys()), value='화학 공정 (기본)', description='업종 선택:', style={'description_width':'80px'}, layout=widgets.Layout(width='300px'))
_change_btn = widgets.Button(description='데이터 변경', button_style='info', layout=widgets.Layout(width='150px',height='32px'))
_sel_out = widgets.HTML()

def _on_change(b):
    name = _selector.value
    _sel_out.value = f'<p style="color:#3B82F6"><b>{name} 데이터 로드 중...</b></p>'
    load_dataset(name)
    rebuild_widgets()
    _sel_out.value = (f'<div style="margin:8px 0;padding:8px 12px;background:#f0fdf4;border-left:4px solid #16a34a;border-radius:6px;font-size:13px">'
        f'<b>{name}</b> 데이터 로드 완료! 아래 위젯 셀을 <b>다시 실행</b>해주세요.</div>')

_change_btn.on_click(_on_change)

print('=' * 50)
print('  준비 완료!')
print('=' * 50)
print(f'  공장 데이터: {len(df)} 로트 ({_selector.value})')
print(f'  모델: factory_model (Random Forest)')
print()
print('  사용 가능한 변수:')
print('    df            -- 공장 데이터 (DataFrame)')
print('    factory_model -- 품질 예측 모델')
xcols = ', '.join(_fi_names[c] for c in input_cols)
ycols = ', '.join(_specs[c][2] for c in output_cols)
print(f'      입력: [{xcols}]')
print(f'      출력: [{ycols}]')
print('    find_optimal  -- 최적 조건 탐색 함수')
print()
print('  모델 성능 (R2):')
for col, score in _r2.items():
    lo,hi,nm = _specs[col]
    bar = '#' * int(score * 20)
    print(f'    {nm:14s} R2={score:.3f}  {bar}')
print()
print('  변수 중요도:')
for idx in np.argsort(_fi)[::-1]:
    col = input_cols[idx]
    bar = '#' * int(_fi[idx] * 40)
    print(f'    {_fi_names[col]:8s} {_fi[idx]:.3f}  {bar}')
print()
display(widgets.HBox([_selector, _change_btn]))
display(_sel_out)


---

### Forward 위젯

슬라이더로 공정조건을 바꿔가며 품질을 예측해보세요.


In [ ]:
forward_ui

### Inverse 위젯

목표 품질을 입력하고 최적 조건을 탐색해보세요.


In [ ]:
inverse_ui